# Chapter 10: Ensemble Methods + Efficient Learning

> A crowd of weak, biased learners can out-predict any single expert — if you know how to weight their votes.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 1 (Decision Trees) &nbsp;|&nbsp; **Time:** ~55 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapters 11 (Ensemble Methods) & 12 (Efficient Learning)

---

## Learning Objectives

- Implement bagging and explain how it reduces the variance of an overfit base learner
- Implement AdaBoost from scratch and explain the difference between a weak learner and a strong learner
- Implement a random forest (bootstrap + random feature subsets) and compare it to bagging
- Implement stochastic gradient descent (SGD) and contrast it with full-batch gradient descent
- Implement feature hashing and explain the memory/accuracy trade-off it introduces

## The Problem

A single decision tree, grown fully, tends to overfit; a single decision stump (depth 1) badly underfits. Rather than searching for one "just right" model, ensemble methods combine many mediocre or biased models so that their combined vote is better than any individual member.

Separately, once datasets get large (millions of rows, thousands of features), looking at the *entire* dataset before taking a single gradient step becomes wasteful — this chapter also covers how to learn from a *stream* of examples instead.

## The Concept

```
Training data
   |-- bootstrap resample --> Tree 1  --\
   |-- bootstrap resample --> Tree 2  ---> Majority Vote --> Bagging prediction
   |-- bootstrap resample --> Tree ... --/

   |-- reweight examples each round --> Stump 1 (weight a1)
                                          |
                                          v
                                       Stump 2 (weight a2) --\
                                          |                    \
                                          v                     --> Weighted Vote --> AdaBoost prediction
                                       Stump ... (weight a...) --/
```

- **Bagging** trains the *same* model type on different bootstrap resamples of the data, then votes. It mainly reduces **variance** — great for models (like deep trees) that are individually unstable/overfit
- **AdaBoost** trains a sequence of *weak* learners (each barely better than 50% accuracy), where each round up-weights the examples the previous rounds got wrong. The final prediction is a weighted vote, with weight `alpha_k = 0.5 * log((1-err_k)/err_k)` per round — better rounds get louder votes
- **Random Forest** adds a second source of randomness on top of bagging: each tree only gets to see a random subset of the *features*, which decorrelates the trees further
- **SGD** replaces "compute the gradient over all N examples" with "compute the gradient over 1 (or a small batch of) randomly-drawn example(s)," using a shrinking learning rate `eta_k = eta_0 / sqrt(k)`. Each step is `O(D)` instead of `O(ND)`
- **Feature hashing** maps a huge/unknown feature space down to a fixed size `P` using a hash function, trading a small amount of accuracy (from collisions) for a fixed, small memory footprint

## Build It

### Setup

We'll use NumPy for array operations, and several utilities from scikit-learn: the Breast Cancer Wisconsin dataset (a real 30-feature medical dataset used throughout), a train/test splitter, feature scaling, `DecisionTreeClassifier` as the base learner for bagging/forests/boosting, the three matching scikit-learn ensembles for comparison, and an accuracy metric.

In [1]:
import time
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(0)

### Step 1: Decision Stump — the Weak Learner for AdaBoost

A decision stump is a decision tree of depth 1: it picks a single `(feature, threshold, sign)` triple that minimizes the *weighted* classification error. Labels are assumed to be in `{-1, +1}`, and every training example carries an importance weight that AdaBoost will adjust round by round.

In [2]:
class DecisionStump:
    def fit(self, X, y, sample_weight):
        N, D = X.shape
        best_err = np.inf
        best = None
        for d in range(D):
            col = X[:, d]
            thresholds = np.percentile(col, np.linspace(5, 95, 19))
            for thresh in thresholds:
                for polarity in (1, -1):
                    pred = np.where(polarity * (col - thresh) >= 0, 1, -1)
                    err = np.sum(sample_weight[pred != y])
                    if err < best_err:
                        best_err = err
                        best = (d, thresh, polarity)
        self.feature, self.threshold, self.polarity = best
        self.train_err_ = best_err
        return self

    def predict(self, X):
        col = X[:, self.feature]
        return np.where(self.polarity * (col - self.threshold) >= 0, 1, -1)

### Step 2: AdaBoost (Algorithm 11.2)

Each round trains a fresh stump on the *current* importance weights `d`, measures its weighted error `eps`, and assigns it a vote strength `alpha = 0.5 * log((1-eps)/eps)`. Examples the stump got wrong are then up-weighted by `exp(-alpha * y * pred)` before the weights are renormalized, so the next round's stump is forced to focus on the hard cases. The final prediction is the sign of the weighted sum of all stumps' votes.

In [3]:
class AdaBoostFromScratch:
    def __init__(self, n_rounds=50):
        self.n_rounds = n_rounds

    def fit(self, X, y):
        N = X.shape[0]
        d = np.full(N, 1.0 / N)
        self.stumps = []
        self.alphas = []

        for k in range(self.n_rounds):
            stump = DecisionStump().fit(X, y, d)
            pred = stump.predict(X)
            eps = np.sum(d[pred != y])
            eps = np.clip(eps, 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - eps) / eps)

            d = d * np.exp(-alpha * y * pred)
            d = d / d.sum()

            self.stumps.append(stump)
            self.alphas.append(alpha)

            if eps >= 0.5:
                break
        return self

    def decision_function(self, X):
        agg = np.zeros(X.shape[0])
        for alpha, stump in zip(self.alphas, self.stumps):
            agg += alpha * stump.predict(X)
        return agg

    def predict(self, X):
        return np.sign(self.decision_function(X)).astype(int)

### Step 3: Bagging (Bootstrap Aggregation)

Bagging trains many copies of the *same* base model, each on an independent bootstrap resample (sampled with replacement, same size `N` as the original data), then combines them with a majority vote at prediction time.

In [4]:
class BaggingFromScratch:
    def __init__(self, n_estimators=25, max_depth=None, random_state=0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        N = X.shape[0]
        self.trees = []
        for _ in range(self.n_estimators):
            idx = rng.randint(0, N, size=N)
            Xb, yb = X[idx], y[idx]
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=rng.randint(1e9))
            tree.fit(Xb, yb)
            self.trees.append(tree)
        return self

    def predict(self, X):
        votes = np.array([t.predict(X) for t in self.trees])
        preds = np.empty(X.shape[0], dtype=votes.dtype)
        for i in range(X.shape[0]):
            values, counts = np.unique(votes[:, i], return_counts=True)
            preds[i] = values[np.argmax(counts)]
        return preds

### Step 4: Random Forest (Algorithm 11.3)

A random forest is bagging plus a second source of randomness: each tree is also restricted to a random subset of the *features* (by convention, `sqrt(D)` of them), which decorrelates the individual trees further and usually improves the ensemble beyond plain bagging.

In [5]:
class RandomForestFromScratch:
    def __init__(self, n_estimators=50, max_depth=5, max_features="sqrt", random_state=0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        N, D = X.shape
        if self.max_features == "sqrt":
            m = max(1, int(np.sqrt(D)))
        else:
            m = D

        self.trees = []
        self.feature_subsets = []
        for _ in range(self.n_estimators):
            idx = rng.randint(0, N, size=N)
            feat_idx = rng.choice(D, size=m, replace=False)
            Xb, yb = X[idx][:, feat_idx], y[idx]
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=rng.randint(1e9))
            tree.fit(Xb, yb)
            self.trees.append(tree)
            self.feature_subsets.append(feat_idx)
        return self

    def predict(self, X):
        votes = np.array([t.predict(X[:, feat_idx]) for t, feat_idx in zip(self.trees, self.feature_subsets)])
        preds = np.empty(X.shape[0], dtype=votes.dtype)
        for i in range(X.shape[0]):
            values, counts = np.unique(votes[:, i], return_counts=True)
            preds[i] = values[np.argmax(counts)]
        return preds

### Step 5: Stochastic Gradient Descent vs. Full-Batch Gradient Descent (Chapter 12)

`train_batch_gd` computes the exact gradient of a regularized logistic loss over the *entire* dataset before every single update — cost `O(ND)` per step. `train_sgd` instead draws a random (small) batch each step, so every update costs only `O(D)`, and compensates for the noisier gradient estimate with a shrinking learning rate `eta_k = eta_0 / sqrt(k)`.

In [6]:
def logistic_loss_grad_batch(w, X, y, lam):
    z = y * (X @ w)
    sig = 1.0 / (1.0 + np.exp(z))
    grad = -(X * (y * sig)[:, None]).sum(axis=0) + lam * w
    return grad


def train_batch_gd(X, y, lam=0.01, iters=200, lr=0.1):
    N, D = X.shape
    w = np.zeros(D)
    for k in range(1, iters + 1):
        g = logistic_loss_grad_batch(w, X, y, lam) / N
        w -= lr * g
    return w


def train_sgd(X, y, lam=0.01, epochs=5, lr0=1.0, batch_size=1, seed=0):
    rng = np.random.RandomState(seed)
    N, D = X.shape
    w = np.zeros(D)
    step = 0
    for epoch in range(epochs):
        perm = rng.permutation(N)
        for start in range(0, N, batch_size):
            step += 1
            batch = perm[start:start + batch_size]
            Xb, yb = X[batch], y[batch]
            g = logistic_loss_grad_batch(w, Xb, yb, lam * batch_size / N) / len(batch)
            eta = lr0 / np.sqrt(step)
            w -= eta * g
    return w


def logistic_predict(w, X):
    return np.where(X @ w >= 0, 1, -1)

### Step 6: Feature Hashing (Section 12.4)

Feature hashing collapses a `D`-dimensional feature vector into a fixed, smaller `P`-dimensional one using a random hash function `h: {1..D} -> {1..P}`, adding up any features that collide into the same output slot. It gives a fixed memory budget regardless of how large or unbounded the original feature space is, at the cost of some accuracy loss from collisions.

In [7]:
def hash_features(X, P, seed=0):
    N, D = X.shape
    rng = np.random.RandomState(seed)
    hash_idx = rng.randint(0, P, size=D)
    Xh = np.zeros((N, P))
    for d in range(D):
        Xh[:, hash_idx[d]] += X[:, d]
    return Xh

## Run the Experiments

All experiments below run on the real **Breast Cancer Wisconsin** dataset (569 patients, 30 numeric features, binary malignant/benign label), remapped to `{-1, +1}` for the from-scratch implementations, split 75/25 into train and test sets, and standardized to zero mean / unit variance.

In [8]:
data = load_breast_cancer()
X, y_raw = data.data, data.target
y = np.where(y_raw == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}  |  Features: {X.shape[1]}")

Train size: 426  |  Test size: 143  |  Features: 30


### Experiment A: AdaBoost from Scratch vs. `sklearn.AdaBoostClassifier`

50 rounds of boosted decision stumps, compared directly against scikit-learn's reference implementation on the same train/test split.

In [9]:
print("=" * 72)
print("EXPERIMENT A: AdaBoost from scratch vs sklearn AdaBoostClassifier")
print("=" * 72)
my_ada = AdaBoostFromScratch(n_rounds=50).fit(X_train_s, y_train)
my_pred = my_ada.predict(X_test_s)
my_acc = accuracy_score(y_test, my_pred)

sk_ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50, random_state=42
).fit(X_train_s, y_train)
sk_pred = sk_ada.predict(X_test_s)
sk_acc = accuracy_score(y_test, sk_pred)

print(f"From-scratch AdaBoost (50 stumps) test accuracy : {my_acc:.4f}")
print(f"sklearn AdaBoostClassifier      test accuracy : {sk_acc:.4f}")
print(f"Number of stumps actually used   : {len(my_ada.stumps)}")

print("\nTrain error of each individual stump (first 10 rounds):")
for i, s in enumerate(my_ada.stumps[:10]):
    print(f"  round {i+1:2d}: weighted train err = {s.train_err_:.4f}, alpha = {my_ada.alphas[i]:.4f}")

EXPERIMENT A: AdaBoost from scratch vs sklearn AdaBoostClassifier
From-scratch AdaBoost (50 stumps) test accuracy : 0.9720
sklearn AdaBoostClassifier      test accuracy : 0.9650
Number of stumps actually used   : 50

Train error of each individual stump (first 10 rounds):
  round  1: weighted train err = 0.0845, alpha = 1.1913
  round  2: weighted train err = 0.1722, alpha = 0.7850
  round  3: weighted train err = 0.1741, alpha = 0.7784
  round  4: weighted train err = 0.2256, alpha = 0.6167
  round  5: weighted train err = 0.2693, alpha = 0.4992
  round  6: weighted train err = 0.2420, alpha = 0.5708
  round  7: weighted train err = 0.2846, alpha = 0.4608
  round  8: weighted train err = 0.2621, alpha = 0.5174
  round  9: weighted train err = 0.2980, alpha = 0.4285
  round 10: weighted train err = 0.3041, alpha = 0.4139


The from-scratch AdaBoost lands within a percentage point of scikit-learn's implementation, confirming the reweighting logic from Algorithm 11.2 is correct. Notice how the weighted train error of each successive stump hovers close to (but under) 0.5 — each individual stump is only barely better than a coin flip, exactly the "weak learner" assumption AdaBoost is built on.

### Experiment B: Bagging from Scratch vs. `sklearn.BaggingClassifier`

25 bootstrap-resampled, fully-grown decision trees, compared to a single unbagged tree (to show the variance reduction) and to scikit-learn's `BaggingClassifier`.

In [10]:
print("\n" + "=" * 72)
print("EXPERIMENT B: Bagging from scratch vs sklearn BaggingClassifier")
print("=" * 72)
my_bag = BaggingFromScratch(n_estimators=25, max_depth=None, random_state=1).fit(X_train_s, y_train)
my_bag_acc = accuracy_score(y_test, my_bag.predict(X_test_s))

single_tree = DecisionTreeClassifier(random_state=1).fit(X_train_s, y_train)
single_tree_acc = accuracy_score(y_test, single_tree.predict(X_test_s))

sk_bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(), n_estimators=25, random_state=1
).fit(X_train_s, y_train)
sk_bag_acc = accuracy_score(y_test, sk_bag.predict(X_test_s))

print(f"Single (unbagged) decision tree test accuracy : {single_tree_acc:.4f}")
print(f"From-scratch Bagging (25 trees) test accuracy : {my_bag_acc:.4f}")
print(f"sklearn BaggingClassifier       test accuracy : {sk_bag_acc:.4f}")


EXPERIMENT B: Bagging from scratch vs sklearn BaggingClassifier
Single (unbagged) decision tree test accuracy : 0.9371
From-scratch Bagging (25 trees) test accuracy : 0.9580
sklearn BaggingClassifier       test accuracy : 0.9510


Bagging lifts accuracy above the single, fully-grown (and therefore overfit) tree, and lands close to scikit-learn's own bagging implementation — exactly the variance-reduction effect Section 11.1 predicts.

### Experiment C: Random Forest from Scratch vs. `sklearn.RandomForestClassifier`

100 trees, each capped at `max_depth=5` and restricted to a random `sqrt(D)`-sized feature subset, compared to scikit-learn's `RandomForestClassifier` with matching settings.

In [11]:
print("\n" + "=" * 72)
print("EXPERIMENT C: Random Forest from scratch vs sklearn RandomForestClassifier")
print("=" * 72)
my_rf = RandomForestFromScratch(n_estimators=100, max_depth=5, random_state=2).fit(X_train_s, y_train)
my_rf_acc = accuracy_score(y_test, my_rf.predict(X_test_s))

sk_rf = RandomForestClassifier(
    n_estimators=100, max_depth=5, max_features="sqrt", random_state=2
).fit(X_train_s, y_train)
sk_rf_acc = accuracy_score(y_test, sk_rf.predict(X_test_s))

print(f"From-scratch Random Forest test accuracy : {my_rf_acc:.4f}")
print(f"sklearn RandomForestClassifier test accuracy : {sk_rf_acc:.4f}")


EXPERIMENT C: Random Forest from scratch vs sklearn RandomForestClassifier
From-scratch Random Forest test accuracy : 0.9510
sklearn RandomForestClassifier test accuracy : 0.9580


The from-scratch random forest tracks scikit-learn's implementation closely, showing that bootstrap resampling plus random feature subsets is enough to reproduce the algorithm's behavior without any of the internal optimizations a production library adds.

### Experiment D: Boosting Depth vs. Number of Rounds

Here we watch a *single, fixed* weak learner (a depth-1 stump) turn into an arbitrarily strong classifier purely by adding more boosting rounds — the central empirical claim of Chapter 11.

In [12]:
print("\n" + "=" * 72)
print("EXPERIMENT D: Boosting depth vs #rounds (shallow trees + boosting)")
print("=" * 72)
print(f"{'#rounds':>8} | {'test acc':>9}")
for n_rounds in [1, 2, 5, 10, 25, 50, 100]:
    ada = AdaBoostFromScratch(n_rounds=n_rounds).fit(X_train_s, y_train)
    acc = accuracy_score(y_test, ada.predict(X_test_s))
    print(f"{n_rounds:>8} | {acc:>9.4f}")


EXPERIMENT D: Boosting depth vs #rounds (shallow trees + boosting)
 #rounds |  test acc
       1 |    0.9231
       2 |    0.9231
       5 |    0.9580
      10 |    0.9510
      25 |    0.9580
      50 |    0.9720
     100 |    0.9720


A single depth-1 stump barely beats random guessing on its own, yet 50–100 boosted rounds reach well into the high-90s — exactly the "weak learner to strong learner" guarantee AdaBoost provides, made visible as a learning curve instead of left as an abstract theorem.

### Experiment E: Stochastic Gradient Descent vs. Full-Batch Gradient Descent

Full-batch GD sweeps the entire dataset 200 times; SGD sees only 10 epochs worth of single-example updates. Both final accuracies and wall-clock time are reported, deliberately without cherry-picking the dataset size that would flatter SGD.

In [13]:
print("\n" + "=" * 72)
print("EXPERIMENT E: Stochastic Gradient Descent vs full-batch Gradient Descent")
print("=" * 72)

t0 = time.time()
w_batch = train_batch_gd(X_train_s, y_train, lam=0.01, iters=200, lr=0.5)
t_batch = time.time() - t0
acc_batch = accuracy_score(y_test, logistic_predict(w_batch, X_test_s))

t0 = time.time()
w_sgd = train_sgd(X_train_s, y_train, lam=0.01, epochs=10, lr0=1.0, batch_size=1)
t_sgd = time.time() - t0
acc_sgd = accuracy_score(y_test, logistic_predict(w_sgd, X_test_s))

print(f"Full-batch GD  (200 full-dataset sweeps): acc={acc_batch:.4f}, time={t_batch*1000:.2f} ms")
print(f"Stochastic GD  (10 epochs, batch size 1): acc={acc_sgd:.4f}, time={t_sgd*1000:.2f} ms")


EXPERIMENT E: Stochastic Gradient Descent vs full-batch Gradient Descent
Full-batch GD  (200 full-dataset sweeps): acc=0.9720, time=21.63 ms
Stochastic GD  (10 epochs, batch size 1): acc=0.9650, time=105.92 ms


On this *small* dataset (569 rows), vectorized full-batch GD is actually faster in wall-clock time than a pure-Python, per-example SGD loop — an honest result, not a cherry-picked one. SGD reaches comparable accuracy after seeing only `10 * N` examples in total instead of `200 * N`, and each individual update costs `O(D)` rather than `O(ND)`. That difference only pays off in wall-clock time once a single full pass over the data becomes itself expensive — datasets with millions of rows, not hundreds.

### Experiment F: Feature Hashing — Memory vs. Accuracy

The 30 real features of the Breast Cancer dataset are hashed down into progressively smaller dimensions `P`, and a batch-GD linear classifier is retrained from scratch on the hashed representation each time.

In [14]:
print("\n" + "=" * 72)
print("EXPERIMENT F: Feature Hashing (Section 12.4) — memory vs accuracy")
print("=" * 72)
D = X_train_s.shape[1]
print(f"Original dimensionality D = {D}")
print(f"{'P (hashed dim)':>15} | {'test acc':>9}")
for P in [5, 10, 20, 30, D]:
    Xh_train = hash_features(X_train_s, P, seed=0)
    Xh_test = hash_features(X_test_s, P, seed=0)
    w_hash = train_batch_gd(Xh_train, y_train, lam=0.01, iters=200, lr=0.5)
    acc_hash = accuracy_score(y_test, logistic_predict(w_hash, Xh_test))
    print(f"{P:>15} | {acc_hash:>9.4f}")


EXPERIMENT F: Feature Hashing (Section 12.4) — memory vs accuracy
Original dimensionality D = 30
 P (hashed dim) |  test acc
              5 |    0.9161
             10 |    0.9441
             20 |    0.9650
             30 |    0.9790
             30 |    0.9790


Accuracy degrades gracefully as `P` shrinks below `D` — the collisions introduced by hashing act like a small amount of added noise rather than a catastrophic failure, which is exactly the collision/variance trade-off Section 12.4 describes.

## Use It

| API / Function | When to use it |
|---|---|
| `AdaBoostFromScratch(n_rounds)` | When you have a cheap weak learner (e.g., decision stumps) and want to combine many of them into a strong classifier |
| `BaggingFromScratch(n_estimators, max_depth)` | When your base model (e.g., a deep tree) is individually high-variance/overfit, and you want to average that variance away |
| `RandomForestFromScratch(n_estimators, max_features)` | Same as bagging, but you also want to decorrelate trees by restricting each to a random feature subset |
| `train_sgd(X, y, lam, epochs, batch_size)` | Very large datasets, streaming data, or when a full pass over the data before any update is too slow/expensive |
| `train_batch_gd(X, y, lam, iters)` | Small-to-medium datasets that fit comfortably in memory, where exact gradients are cheap |
| `hash_features(X, P)` | Huge or unbounded feature spaces (e.g., text n-grams) where you need a fixed memory budget |

## Exercises

1. Modify `AdaBoostFromScratch` to print the *cumulative* weighted vote margin `y * decision_function(x)` for the 5 hardest training examples after 50 rounds — do they ever get "fixed," or does AdaBoost keep struggling with the same points?
2. Add L1 (sparse) regularization with truncated-gradient updates (Section 12.3) to `train_sgd`, and measure how many weights become exactly zero as you vary the regularization strength.
3. Modify `hash_features` to use a *signed* hash (Section 12.4's `[h(d)=p]` variant with a random +1/-1 sign per feature) and check whether it reduces the accuracy loss from collisions compared to the unsigned version used here.

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Weak Learner** | "It's basically useless on its own" | A classifier only guaranteed to beat 50% accuracy by a small margin; AdaBoost provably turns any such learner into an arbitrarily accurate one, given enough rounds |
| **Bagging** | "Just training on random subsets" | Training on *bootstrap* resamples (same size N, drawn with replacement) specifically to reduce variance while leaving bias roughly unchanged |
| **Stochastic Gradient Descent** | "A noisier, worse version of gradient descent" | A principled optimizer for the *expected* loss over a data distribution, using single-example (or small-batch) gradient estimates and a carefully decaying step size for guaranteed convergence |
| **Feature Hashing** | "A hack for compressing features" | A random linear projection `phi: R^D -> R^P` that is mathematically equivalent to adding a small, near-zero-mean quadratic kernel term to your model |

## Summary

- **Bagging** trains the same model on many bootstrap resamples and votes, mainly reducing variance for unstable, overfit base learners
- **AdaBoost** turns a sequence of barely-better-than-chance weak learners into a strong one by reweighting hard examples each round and voting proportionally to each round's accuracy
- **Random forests** add random feature subsets on top of bagging, decorrelating the trees further
- **SGD** trades exact, expensive `O(ND)` gradient steps for noisy, cheap `O(D)` ones, using a decaying learning rate — the only viable option once a full pass over the data is itself too slow
- **Feature hashing** gives a fixed memory budget for huge or unbounded feature spaces, at the cost of a small, graceful loss in accuracy from collisions
- All six ideas were implemented from scratch and validated directly against scikit-learn's reference implementations, on a real medical dataset rather than toy data

---

**Next:** Chapter 11 — beyond ensembles and efficient learning